# Maternal Health Risk Stratification – v4 (Label-noise aware)

**StudyBuild Project 01**

Key change in v4:  
- Explicit removal of **true label conflicts** (identical features but different RiskLevel).  
- Exact duplicates and HeartRate anomalies also removed.  
- Final clean size = **380** records.

Still 3-class (Low / Mid / High). Pipeline + `class_weight="balanced"` kept from previous version.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split

from src.model import (
    FEATURES, RISK_ORDER, load_and_clean,
    make_logistic_pipeline, tune_tree
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

## Q1 – Load & rigorous cleaning (including label-noise removal)

In [ ]:
df_raw = pd.read_csv("../data/Maternal Health Risk Data Set.csv")
print("Raw shape:", df_raw.shape)
print("Exact full-row duplicates:", df_raw.duplicated().sum())

# Full cleaning (HeartRate filter + label conflicts + exact duplicates)
df_clean = load_and_clean("../data/Maternal Health Risk Data Set.csv")
print("\nCleaned shape:", df_clean.shape)
print("\nClass distribution:")
print(df_clean["RiskLevel"].value_counts())
print(df_clean["RiskLevel"].value_counts(normalize=True).round(3))
print("\ndtype:", df_clean["RiskLevel"].dtype)

df_clean.to_csv("../data/maternal_health_clean.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
counts = df_clean["RiskLevel"].value_counts().reindex(RISK_ORDER)
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar([c.title() for c in counts.index], counts.values, color=colors, edgecolor="black")
ax.set_title("Cleaned Risk Level Distribution (N = 380)\nAfter removing label conflicts + exact duplicates", fontweight="bold")
ax.set_ylabel("Number of Patients")
for bar, val in zip(bars, counts.values):
    pct = val / len(df_clean) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+2, f"{val}\n({pct:.1f}%)",
            ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q1_risk_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### Cleaning decisions (v4)

| Step | Action | Records affected |
|------|--------|------------------|
| 1 | HeartRate < 40 removed | 2 |
| 2 | **True label conflicts** removed (same features, different RiskLevel) | 215 rows (35 feature groups) |
| 3 | Exact duplicates removed | remaining duplicates |
| **Final** | Clean dataset | **380** |

Rationale: Keeping rows that have identical clinical measurements but contradictory risk labels forces the model to learn noise. Removing them produces a cleaner educational baseline.

## Q2 – Feature distributions by risk group

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, feat in enumerate(FEATURES):
    sns.boxplot(x="RiskLevel", y=feat, data=df_clean, order=RISK_ORDER, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{feat} by Risk Level", fontweight="bold")
    axes[i].set_xlabel("")
plt.suptitle("Clinical Feature Distributions by Risk Level (Clean Data N=380)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../figures/q2_risk_feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()
print(df_clean.groupby("RiskLevel", observed=True)[FEATURES].agg(["mean", "median"]).round(2))

## Q3 – Correlation & association with High Risk

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_clean[FEATURES].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Feature Correlation Matrix (Pearson)", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q3_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

df_clean["IsHighRisk"] = (df_clean["RiskLevel"] == "high risk").astype(int)
print(pd.DataFrame({
    "Pearson": df_clean[FEATURES].corrwith(df_clean["IsHighRisk"]),
    "Spearman": df_clean[FEATURES].corrwith(df_clean["IsHighRisk"], method="spearman"),
}).sort_values("Pearson", ascending=False).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(data=df_clean, x="SystolicBP", y="BS", hue="RiskLevel", hue_order=RISK_ORDER,
                palette={"low risk":"#2ecc71","mid risk":"#f39c12","high risk":"#e74c3c"},
                alpha=0.7, s=60, ax=ax)
ax.set_title("Blood Sugar vs Systolic BP by Risk Level", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q3_scatter_bs_vs_sysbp.png", dpi=300, bbox_inches="tight")
plt.show()

## Q4 & Q5 – Models (Pipeline + balanced + limited GridSearch)

In [ ]:
X = df_clean[FEATURES]
y = df_clean["RiskLevel"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Logistic Regression (Pipeline)
pipe_lr = make_logistic_pipeline(RANDOM_STATE)
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

# Decision Tree with limited GridSearch
pipe_dt, best_params, best_cv = tune_tree(X_train, y_train, RANDOM_STATE)
print("Best DT params:", best_params)
print("Best CV macro-recall:", round(best_cv, 3))
y_pred_dt = pipe_dt.predict(X_test)

print("\n=== LOGISTIC REGRESSION (balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, digits=3))

print("\n=== DECISION TREE (tuned, balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt, digits=3))

hr_recall = recall_score(y_test, y_pred_dt, labels=["high risk"], average=None)[0]
print(f"\n>>> High-risk Recall (tuned DT): {hr_recall:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, title, cmap in [
    (axes[0], y_pred_lr, "Logistic Regression (balanced)", "Blues"),
    (axes[1], y_pred_dt, "Decision Tree (tuned, balanced)", "Greens"),
]:
    cm = confusion_matrix(y_test, pred, labels=RISK_ORDER)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=[r.title() for r in RISK_ORDER],
                yticklabels=[r.title() for r in RISK_ORDER])
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("../figures/q4_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

## Q6 – Error analysis

In [ ]:
test_df = X_test.copy()
test_df["Actual"] = y_test.values
test_df["Predicted"] = y_pred_dt
test_df["Error"] = test_df["Actual"] != test_df["Predicted"]
mis = test_df[test_df["Error"]]
error_pairs = (mis.groupby(["Actual", "Predicted"], observed=True).size()
               .reset_index(name="Count").sort_values("Count", ascending=False))
error_pairs["Share_%"] = (error_pairs["Count"] / max(len(mis), 1) * 100).round(1)
print(error_pairs)

fig, ax = plt.subplots(figsize=(8, 4.5))
ylabels = error_pairs.apply(lambda r: f"{r['Actual'].title()} → {r['Predicted'].title()}", axis=1)
sns.barplot(x="Count", y=ylabels, data=error_pairs, palette="Reds_r", ax=ax)
ax.set_title("Distribution of Misclassification Errors", fontweight="bold")
ax.set_xlabel("Number of Misclassified Test Cases")
plt.tight_layout()
plt.savefig("../figures/q6_error_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Q7 – Feature importance & LR coefficients

In [ ]:
dt_clf = pipe_dt.named_steps["clf"]
imp_dt = pd.DataFrame({"Feature": FEATURES, "Importance": dt_clf.feature_importances_}).sort_values("Importance", ascending=False)
print(imp_dt)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=imp_dt, x="Importance", y="Feature", palette="Blues_r", ax=ax)
ax.set_title("Decision Tree Feature Importance", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q7_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

lr_clf = pipe_lr.named_steps["clf"]
coef_df = pd.DataFrame(lr_clf.coef_, index=lr_clf.classes_, columns=FEATURES)
print(coef_df.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
high_coef = coef_df.loc["high risk"].sort_values()
colors_c = ["#e74c3c" if v > 0 else "#3498db" for v in high_coef]
high_coef.plot(kind="barh", color=colors_c, ax=ax, edgecolor="black")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Logistic Regression Coefficients – High-Risk Class", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q7_lr_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()

## Q8 – Clinical take-aways

In [ ]:
print(f'''
1. Cleaning: Removed 2 HeartRate anomalies + 215 label-conflict rows + exact duplicates → N=380.
2. Primary biomarkers remain BS and SystolicBP.
3. Model is still only a preliminary screening aid – not a diagnostic tool.
4. Limitations: small clean sample, single geography, missing important covariates.
''')

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=imp_dt, x="Importance", y="Feature", color="teal", ax=ax)
ax.set_title("Executive Overview – Key Clinical Biomarkers", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q8_executive_summary_chart.png", dpi=300, bbox_inches="tight")
plt.show()

---
**End of v4 analysis.**  
True label conflicts have now been explicitly removed in addition to exact duplicates.